# Cell 1: Imports & Setup

In [2]:
import torch
import os
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from dotenv import load_dotenv
load_dotenv()

# 1. Mount Drive

# 2. Authentication

# 3. Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


# Cell 2: Configuration & Directories

In [7]:
# --- Configuration ---
CSV_PATH = "./RepliQA_long_Test.csv"
OUTPUT_DIR = "./RepliQA_long_Test_vectors"

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Vectors will be saved to: {OUTPUT_DIR}")

# Define the Prompt Template
def make_prompt(sentence):
    return f"Tell me about {sentence}"

# --- Model Config ---
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
TARGET_LAYER_INDEX = 16  # We want the INPUT to Layer 16
# (In HF hidden_states tuple: Index 0=Embeddings, Index 1=Layer0 Output... Index 16 = Layer15 Output)

Vectors will be saved to: ./RepliQA_long_Test_vectors


# Cell 3: Load Model & Tokenizer

In [4]:
print("Loading Model...")

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Load Model (Frozen, bfloat16 for efficiency)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
model.eval()

print("Model Loaded.")

Loading Model...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model Loaded.


# Cell 4: Extraction Loop

In [8]:
# 1. Load Data
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV not found at: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)

# Check columns
required_cols = ['question_id', 'long_answer']
if not all(col in df.columns for col in required_cols):
    raise ValueError(f"CSV must contain columns: {required_cols}")

print(f"Found {len(df)} rows. Starting extraction...")

# 2. Calculate Prefix Length (To remove "Tell me about " tokens)
# We tokenize the empty template to see how many tokens the prefix uses
prefix_text = "Tell me about "
prefix_tokens = tokenizer(prefix_text, add_special_tokens=True)["input_ids"]
# Note: Llama 3.1 sometimes adds a start token.
# We will use a dynamic method inside the loop for higher precision.

# 3. Loop
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting"):
    q_id = str(row['question_id'])
    sentence = str(row['long_answer'])

    # Setup Filename
    # We clean the ID to be safe for filenames
    safe_id = "".join([c for c in q_id if c.isalnum() or c in ('-','_')])
    save_path = os.path.join(OUTPUT_DIR, f"vec_{safe_id}.pt")

    # Skip if already exists (resume capability)
    if os.path.exists(save_path):
        continue

    # Prepare Prompt
    full_prompt = make_prompt(sentence)

    # Tokenize
    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)

    # Calculate where the sentence actually starts
    # We re-tokenize just the sentence to know its length roughly,
    # but exact slicing is safer by subtracting the prefix length.
    total_len = inputs['input_ids'].shape[1]

    # Heuristic: Tokenize prefix without special tokens to get count
    prefix_ids = tokenizer(prefix_text, add_special_tokens=False)["input_ids"]
    prefix_len = len(prefix_ids)

    # Note: Llama 3 often adds a <begin_of_text> token (ID 128000) at index 0.
    # So the prefix "Tell me about " usually starts at index 1.
    # We will slice from (prefix_len + 1) to keep only the sentence.
    start_index = prefix_len + 1

    # Run Model
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # Extract Hidden States
    # Index 16 in 'hidden_states' is the Output of Layer 15 (which is Input to Layer 16)
    layer_output = outputs.hidden_states[TARGET_LAYER_INDEX]

    # layer_output shape: [1, Total_Seq_Len, 4096]

    # Slice: Remove "Tell me about " -> Keep "{Sentence}"
    # We check if start_index is valid, otherwise take whole thing
    if start_index < total_len:
        sentence_vectors = layer_output[0, start_index:, :].clone()
    else:
        # Fallback if sentence is extremely short or empty
        sentence_vectors = layer_output[0, :, :].clone()

    # Move to CPU and Save
    # We save a dictionary with metadata just in case
    data_to_save = {
        "question_id": q_id,
        "sentence_text": sentence,
        "vectors": sentence_vectors.cpu().float() # Save as float32 for compatibility
    }

    torch.save(data_to_save, save_path)

print(f"\nDone! Vectors saved to {OUTPUT_DIR}")

Found 7097 rows. Starting extraction...


Extracting:   0%|          | 0/7097 [00:00<?, ?it/s]


Done! Vectors saved to ./RepliQA_long_Test_vectors


# Cell 5: Verification (Optional)

In [9]:
# Check the first file found
files = os.listdir(OUTPUT_DIR)
if files:
    test_file = os.path.join(OUTPUT_DIR, files[0])
    data = torch.load(test_file)

    print(f"--- Verification ---")
    print(f"Loaded: {files[0]}")
    print(f"ID: {data['question_id']}")
    print(f"Text: '{data['sentence_text']}'")
    print(f"Vector Shape: {data['vectors'].shape}")
    print(f"   -> (Tokens in sentence, Hidden Dim 4096)")
else:
    print("No files found yet.")

--- Verification ---
Loaded: vec_lmksiicv-q1.pt
ID: lmksiicv-q1
Text: 'Our journey into local innovation begins at the nexus of necessity and ingenuity: the college campus. Here, surrounded by academic resources and fueled by the fervor of youthful ambition, student entrepreneurs like Michael Chen have managed to create remarkable inventions between classes. On October 15, 2023, Chen unveiled "EcoVent," a smart ventilation system designed to reduce energy consumption in academic buildings. The AI-driven tech can potentially cut down energy usage by an astonishing 20%, showcasing the practical impact of dorm-room ideation.'
Vector Shape: torch.Size([103, 4096])
   -> (Tokens in sentence, Hidden Dim 4096)


In [ ]:
import os
import torch
from tqdm.auto import tqdm

# Configuration
OUTPUT_DIR = "./Entities_Train_vectors"

print(f"Scanning files in: {OUTPUT_DIR}")

max_len = 0
max_id = None
max_text = ""

files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(".pt")]

if not files:
    print("No vector files found.")
else:
    for f in tqdm(files, desc="Checking lengths"):
        path = os.path.join(OUTPUT_DIR, f)

        try:
            # Load data (map to CPU to be fast and save GPU memory)
            data = torch.load(path, map_location="cpu")

            # shape is [Seq_Len, 4096]
            # We want dimension 0 (the number of tokens)
            curr_len = data['vectors'].shape[0]

            if curr_len > max_len:
                max_len = curr_len
                max_id = data.get('question_id', 'Unknown')
                max_text = data.get('sentence_text', '')

        except Exception as e:
            print(f"Error reading {f}: {e}")

    print("\n" + "="*40)
    print(f"MAXIMUM VECTORS FOUND: {max_len}")
    print(f"Question ID: {max_id}")
    print(f"Text Snippet: {max_text[:100]}...")
    print("="*40)

Scanning files in: ./Entities_Train_vectors


Checking lengths:   0%|          | 0/1668 [00:00<?, ?it/s]

In [1]:
import os
import json
import torch
import torch.nn as nn
import pandas as pd
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from dataclasses import dataclass

# Gemini Imports
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

from dotenv import load_dotenv
load_dotenv()

# 1. Mount Drive

# 2. Authentication

# Initialize Gemini Client
gemini_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# 3. Hardware check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
import os
import torch
from tqdm.auto import tqdm

def compile_vectors(vectors_dirs, output_path):
    compiled_vectors = {}

    for directory in vectors_dirs:
        if not os.path.exists(directory):
            print(f"Directory not found: {directory}")
            continue

        print(f"Processing {directory}...")
        for filename in tqdm(os.listdir(directory)):
            if filename.endswith('.pt'):
                # Extract the safe_id from the filename (e.g., "vec_123.pt" -> "123")
                safe_id = filename[4:-3]
                file_path = os.path.join(directory, filename)

                try:
                    # Load directly to CPU to save RAM during compilation
                    loaded_data = torch.load(file_path, map_location="cpu", weights_only=False)
                    # Store only the tensor in the dictionary
                    compiled_vectors[safe_id] = loaded_data['vectors']
                except Exception as e:
                    print(f"Error loading {filename}: {e}")

    print(f"Saving {len(compiled_vectors)} total vectors to {output_path}...")
    torch.save(compiled_vectors, output_path)
    print("✅ Compilation complete!")

# Define your test directories
test_vectors_dirs = (
    "./RepliQA_Test_vectors",
    "./RepliQA_long_Test_vectors",
    "./FaithEval_test_long_vectors",
    "./FaithEval_test_short_vectors",
    "./Real_Test_vectors"
)

# Output path for the compiled test dictionary
compiled_test_path = "./compiled_test_vectors.pt"

# Run the compiler
compile_vectors(test_vectors_dirs, compiled_test_path)

Processing ./RepliQA_Test_vectors...


  0%|          | 0/2383 [00:00<?, ?it/s]

Processing ./RepliQA_long_Test_vectors...


  0%|          | 0/7097 [00:00<?, ?it/s]

Processing ./FaithEval_test_long_vectors...


...

Processing ./FaithEval_test_short_vectors...


  0%|          | 0/200 [00:00<?, ?it/s]

Processing ./Real_Test_vectors...


  0%|          | 0/151 [00:00<?, ?it/s]

Saving 10031 total vectors to ./compiled_test_vectors.pt...
✅ Compilation complete!


In [2]:
!ls -lh "./All_Train_Vectors.zip"

ls: cannot access './All_Train_Vectors.zip': No such file or directory
